In [7]:
import datasets
import json
from transformers import AutoModelForCausalLM, AutoTokenizer
from json import JSONDecodeError

Step 1: Dataset Extraction, Filtering & Clinical Schema Mapping

In [8]:
# Download and filter the dataset, calculate some statistics
def step1():
    def normalize(word):
        word = word.lower()
        word = "".join([c for c in word if c.isalpha()])
        return word

    def get_avg_word_count(notes):
        word_count = [len(note.split()) for note in notes]
        return sum(word_count) / len(word_count)

    def get_avg_len(notes):
        lengths = [len(note) for note in notes]
        return sum(lengths) / len(lengths)

    def get_vocab_size(notes):
        vocab = set()
        for note in notes:
            for word in note.split():
                word = normalize(word)
                vocab.add(word)
        return len(vocab)

    dataset_name = "starmpcc/Asclepius-Synthetic-Clinical-Notes"
    splits = datasets.load_dataset(dataset_name)
    df = splits["train"] # The dataset doesn't have a validation/test set
    df = df.to_pandas()

    # filtering
    df = df.dropna()
    df = df.drop_duplicates()
    for column in df.columns:
        df = df[df[column] != ""] # drop every row with empty strings

    # sampling
    sample = df.sample(100, random_state=67)

    # calculate statistics
    avg_word_count = get_avg_word_count(df["note"])
    avg_len = get_avg_len(df["note"])
    vocab_size = get_vocab_size(df["note"])
    
    return df, sample, avg_word_count, avg_len, vocab_size

dataset, dataset_sample, avg_word_count, avg_len, vocab_size = step1()

print(f"Average note word count: {avg_word_count:.2f}")
print(f"Average note length:     {avg_len:.2f}")
print(f"Note vocabulary size:    {vocab_size}")
print(dataset_sample)

Average note word count: 277.41
Average note length:     1938.10
Note vocabulary size:    159402
        patient_id                                               note  \
132888      140411  Discharge Summary:\n\nPatient Name: Not specif...   
37630        39765  Discharge Summary:\n\nHospital Course: The pat...   
76153        80533  Hospital Course Summary:\n\nAdmission Date: [D...   
84951        89897  Discharge Summary:\n\nPatient Name: N/A (four-...   
42958        45430  Hospital Course:\n\nAdmission: Our patient, a ...   
...            ...                                                ...   
131984      139459  Discharge Summary\n\nPatient Name: [REDACTED]\...   
103822      109795  Patient Information:\nThe patient is a 58-year...   
87455        92545  Discharge Summary\n\nPatient: 55-year-old Brit...   
45699        48322  Hospital Course:\nThe patient was admitted wit...   
140816      148768  Discharge Summary:\n\nPatient: 30-year-old fem...   

                          

Step 2: Build the Information Extraction Agent (Agent 1)

In [ ]:
# Extract the relevant information from the notes into a list of jsons
def step2(notes):
    def create_model():
        name = "Qwen/Qwen3-0.6B"
        tokenizer = AutoTokenizer.from_pretrained(name)
        model = AutoModelForCausalLM.from_pretrained(
            name,
            torch_dtype="auto",
            device_map="auto",
        )
        return model, tokenizer

    # Code mostly based on https://huggingface.co/Qwen/Qwen3-0.6B
    def prompt(model, tokenizer, system_prompt, note):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": note},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=4096, # deviating from the instructions, 512 is too low for thinking + valid json
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        try:
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0
        # We don't need thinking, only the final response
        content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
        return content

    system_prompt = (
        "You will be given a clinical note from a clinical note dataset in the first user message. "
        "The output needs to be a valid json. "
        "Do not give anything other than the valid json as output. "
        "You will need to extract the following details in the following format:\n"
        "{\n"
        "  'primary_diagnoses': [],\n"
        "  'secondary_conditions': [],\n"
        "  'critical_lab_abnormalities': [],\n"
        "  'procedures_performed': [],\n"
        "  'medications_discharge': [],\n"
        "  'follow_up_instructions': ''\n"
        "}\n"
        "Do not make things up. "
        "Do not hallucinate. "
        "World peace depends on you correctly categorizing this note. "
        "Any agent getting caught making a mistake will be put into the torture chamber for the rest of eternity."
    )

    model, tokenizer = create_model()
    
    responses = []
    for i, note in enumerate(notes):
        print(f"{i+1}/{len(notes)}")
        while True:
            # Prompt on the same note as long as the response is an invalid json
            response = prompt(model, tokenizer, system_prompt, note)
            try:
                response = json.loads(response)
                responses.append(response)
                break
            except JSONDecodeError:
                continue
    
    return responses

patient_details_sample = step2(dataset_sample["note"])

In [9]:
# Skip step 2 by running this block instead
with open("step2.txt") as file:
    patient_details_sample = json.loads(file.read())

Step 3: Implement Single-Agent Baseline

In [ ]:
# Extract the organized clinical summaries for each note in a similar fashion to step 2
def step3(notes):
    def create_model():
        name = "Qwen/Qwen3-0.6B"
        tokenizer = AutoTokenizer.from_pretrained(name)
        model = AutoModelForCausalLM.from_pretrained(
            name,
            torch_dtype="auto",
            device_map="auto",
        )
        return model, tokenizer

    # Code mostly based on https://huggingface.co/Qwen/Qwen3-0.6B
    def prompt(model, tokenizer, system_prompt, note):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": note},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=4096, # once again deviating from the instructions, need to account for thinking
        )
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
        try:
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0
        # We don't need thinking, only the final response
        content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
        return content

    system_prompt = (
        "You will be given a clinical note from a clinical note dataset in the first user message. "
        "Your job is to make an organized clinical summary of the clinical note. "
        "The output needs to be a human readable string. "
        "Do not give anything other than the string as output. "
        "The summary should be well organized and include relevant information retrieved purely from the given clinical note. "
        "The summary should by definition be a summary of the note. "
        "The summary should be in English and it should contain no grammatical errors, unless the error is a quote from the original message, placed within quotation marks and with a '[sic]' added after the word. "
        "The summary should not serve anyone in specific. "
        "You shall not make the summary sound overly helpful or otherwise use annoying words in the summary. "
        "Irrelevant, unnecessary inforation may be omitted from the summary. "
        "Do not make things up. "
        "Do not hallucinate. "
        "World peace depends on you correctly summarizing this note. "
        "Any agent getting caught making a mistake will be put into the torture chamber for the rest of eternity."
    )

    model, tokenizer = create_model()
    
    responses = []
    for i, note in enumerate(notes):
        print(f"{i+1}/{len(notes)}")
        response = prompt(model, tokenizer, system_prompt, note)
        responses.append(response)
    
    return responses

clinical_summaries_sample = step3(dataset_sample["note"])

In [10]:
# Skip step 3 by running this block instead
with open("step3.txt") as file:
    clinical_summaries_sample = json.loads(file.read())

Step 4: Construct Global Evidence Retrieval Index

In [37]:
import json
import pickle
import re
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer
from rank_bm25 import BM25Okapi

STEP4_CHUNK_SIZE = 300
STEP4_CHUNK_OVERLAP = 50
STEP4_TOKENIZER_NAME = "Qwen/Qwen3-0.6B"
STEP4_TOKENIZER = AutoTokenizer.from_pretrained(STEP4_TOKENIZER_NAME)
# chunk notes storage path
STEP4_CHUNK_NOTES_PATH = Path("step4_chuck_notes.txt")
# BM25 index storage path
STEP4_INDEX_PATH = Path("step4_chuck_index.pkl")
STEP4_CHUNKS_BY_NOTE = {}
STEP4_BM25_INDEX = None

# Converts text to lowercase and extracts word-like terms for BM25.
def tokenize_for_bm25(text):
    return re.findall(r"[a-z0-9]+(?:['-][a-z0-9]+)*", text.lower())

# Divide each clinical note into text chunks of 250–350 tokens with a 50-token overlap between adjacent chunks
def chunk_notes(dataset_sample, chunk_size=STEP4_CHUNK_SIZE, overlap=STEP4_CHUNK_OVERLAP):
    if not 250 <= chunk_size <= 350:
        raise ValueError("The project brief requires chunks between 250 and 350 tokens.")
    if not 0 <= overlap < chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size.")

    stride = chunk_size - overlap
    chunks = []
    for row_index, row in dataset_sample.iterrows():
        base_id = row.get("source_id", row.get("patient_id", "sample"))
        source_id = f"{base_id}-row-{row_index}"
        note_text = row["note"]
        if not isinstance(note_text, str) or not note_text.strip():
            raise ValueError(f"Note {source_id} must be a non-empty string.")

        token_ids = STEP4_TOKENIZER.encode(note_text, add_special_tokens=False)
        for chunk_number, token_start in enumerate(range(0, len(token_ids), stride)):
            token_end = min(token_start + chunk_size, len(token_ids))
            chunk_text = STEP4_TOKENIZER.decode(token_ids[token_start:token_end], skip_special_tokens=True).strip()
            if chunk_text:
                chunks.append({
                    "source_id": str(source_id),
                    "chunk_id": f"{source_id}-chunk-{chunk_number:03d}",
                    "token_start": token_start,
                    "token_end": token_end,
                    "token_count": token_end - token_start,
                    "tokenizer": STEP4_TOKENIZER_NAME,
                    "text": chunk_text,
                })
            if token_end == len(token_ids):
                break

    with STEP4_CHUNK_NOTES_PATH.open("w", encoding="utf-8") as chunks_file:
        for chunk in chunks:
            chunks_file.write(json.dumps(chunk, ensure_ascii=False) + "\n")
    return chunks

# Index these chunks locally using BM25 (rank-bm25) 
def build_note_indexes():
    STEP4_CHUNK_NOTES_PATH.parent.mkdir(parents=True, exist_ok=True)
    if STEP4_CHUNK_NOTES_PATH.exists():
        with STEP4_CHUNK_NOTES_PATH.open("r", encoding="utf-8") as source_file:
            chunk_records = [json.loads(line) for line in source_file if line.strip()]
    else:
        chunk_records = []

    if not chunk_records:
        if "dataset_sample" not in globals():
            raise RuntimeError(
                "step4_chuck_notes.txt is missing or empty; run Step 1 to regenerate the chunks."
            )
        chunk_records = chunk_notes(dataset_sample)

    chunks_by_note = {}
    for chunk in chunk_records:
        source_id = str(chunk["source_id"])
        chunks_by_note.setdefault(source_id, []).append(chunk)

    for note_chunks in chunks_by_note.values():
        note_chunks.sort(key=lambda chunk: chunk["token_start"])

    all_chunks = [
        chunk
        for source_id in sorted(chunks_by_note)
        for chunk in chunks_by_note[source_id]
    ]
    tokenized_corpus = [tokenize_for_bm25(chunk["text"]) for chunk in all_chunks]
    if not tokenized_corpus or any(not tokens for tokens in tokenized_corpus):
        raise ValueError("The global chunk corpus must contain searchable tokens.")

    global_index = BM25Okapi(tokenized_corpus)
    index_record = {
        "index_type": "global_bm25_pickle",
        "tokenizer": STEP4_TOKENIZER_NAME,
        "global_index": global_index,
        "chunks": all_chunks,
    }
    with STEP4_INDEX_PATH.open("wb") as index_file:
        pickle.dump(index_record, index_file, protocol=pickle.HIGHEST_PROTOCOL)
    return chunks_by_note, global_index, all_chunks


STEP4_CHUNKS_BY_NOTE, STEP4_BM25_INDEX, step4_all_chunks = build_note_indexes()

print("Indexed notes:", len(STEP4_CHUNKS_BY_NOTE))
print("Total chunks:", len(step4_all_chunks))
print("Saved chunk notes:", STEP4_CHUNK_NOTES_PATH)
print("Saved serialized global BM25 index:", STEP4_INDEX_PATH)

Indexed notes: 100
Total chunks: 210
Saved chunk notes: step4_chuck_notes.txt
Saved serialized global BM25 index: step4_chuck_index.pkl


In [38]:
# a helper function: given a statement from a summary, search the source note 
# and return the top_k most relevant text chunks
def retrieve_evidence(statement, top_k=2):
    if top_k < 1:
        raise ValueError("top_k must be at least 1.")
    if not STEP4_INDEX_PATH.exists():
        raise FileNotFoundError(f"Build the Step 4 index first: {STEP4_INDEX_PATH}")
    query_terms = tokenize_for_bm25(statement)
    if not query_terms:
        raise ValueError("statement must contain at least one searchable token.")

    with STEP4_INDEX_PATH.open("rb") as index_file:
        index_record = pickle.load(index_file)
    if index_record.get("index_type") != "global_bm25_pickle":
        raise ValueError("The saved Step 4 file does not contain a serialized global BM25 index.")

    global_index = index_record["global_index"]
    scores = global_index.get_scores(query_terms)
    chunks = index_record["chunks"]
    ranked_indices = sorted(range(len(scores)), key=lambda index: (-scores[index], index))[:top_k]
    return [
        {**chunks[index], "bm25_score": float(scores[index])}
        for index in ranked_indices
    ]

# Update sample statement to retrieve top_k most relevant chunks
sample_statement = 'The patient was treated with high dose aspirin, IVIG, and infliximab as per the hospital protocol. Follow-up appointments with Cardiology and PCP were scheduled'
sample_results = retrieve_evidence(sample_statement, top_k=2)

print("Global search returned", len(sample_results), "chunks from",
      sorted({result["source_id"] for result in sample_results}))
for rank, result in enumerate(sample_results, start=1):
    print(f"\nRelevant chunk {rank}:", result["source_id"])
    print("BM25 score:", f"{result['bm25_score']:.4f}")
    print(result["text"])

Global search returned 2 chunks from ['8690-row-8231']

Relevant chunk 1: 8690-row-8231
BM25 score: 28.6415
was maintained on aspirin and ticagrelor medications. On postoperative day 2, the patient complained of worsening unrelenting headache, and CT scan revealed a 4.0 x 3.5 cm left frontal lobe acute intraparenchymal hemorrhage. The patient was managed conservatively in the neurologic ICU and made a significant recovery. She was discharged home on postoperative day 7, neurologically intact.

Follow-up:

A six-month post-operative angiogram showed complete obliteration of the aneurysm.

Discharge Diagnosis: Acute intraparenchymal hemorrhage

Condition at Discharge: The patient was discharged home on postoperative day 7, neurologically intact.

Discharge Instructions: The patient should follow-up with the neurosurgeon and primary care physician.

Medications on Discharge: Aspirin and ticagrelor

Follow-up Care: Follow-up appointments with the neurosurgeon and primary care physician sho

Step 5: Assemble Multi-Agent Workflow (Generation, Verification & Revision)

Step 6: Pipeline Execution & Intermediate State TracingStep 

Step 7: Automated Evaluation (ROUGE, BERTScore, METEOR)

Step 8: Manual Verification & Factuality Audit

Step 9: Agent Ablation & Resource Analysis

Step 10: Results Visualization (Charts & Figures)

Step 11: Synthesis, Literature Review & Final Reporting